In [ ]:

def calculate_binary_information_value(feature: numpy.ndarray, target: numpy.ndarray, bins: int) -> float:
    quantiles = numpy.linspace(0, 1, bins + 1)
    bins = numpy.unique(numpy.quantile(feature, quantiles))
    bins[0] -= 0.0001
    bins[-1] += 0.0001
    
    feature = numpy.digitize(feature, bins, right = True)
    n, events = [ ], [ ]
    for bin in range(1, len(bins)):
        mask = (feature == bin)
        n.append(mask.sum())
        events.append((target * mask).sum())
    n = numpy.array(n)
    events = numpy.array(events)

    non_events = n - events
    events_prc = numpy.maximum(events, 0.5) / events.sum()
    non_events_prc = numpy.maximum(non_events, 0.5) / non_events.sum()

    woe = numpy.log(events_prc / non_events_prc)
    iv = woe * (events_prc - non_events_prc)
    return iv.sum()


In [ ]:
import pandas
data = pandas.read_csv("churn.csv")

data = data.drop(columns = [ 'customerID' ])
data = data[data["TotalCharges"] != ' ']
data["TotalCharges"] = data["TotalCharges"].astype(float)

data

In [ ]:
import sklearn.model_selection

X = data.drop(columns = [ "Churn" ])
y = (data["Churn"] == "Yes").to_numpy()

X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size = 0.1, stratify = y, random_state = 1
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
numeric = [ "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges" ]
categorical = list(set(X_train.columns) - set(numeric))

len(numeric), len(categorical), len(X_train.columns)

In [ ]:
import sklearn.compose
import sklearn.preprocessing

ct = sklearn.compose.ColumnTransformer(
    transformers=[
        ("numeric", 'passthrough', numeric),
        ("categorical", sklearn.preprocessing.OneHotEncoder(drop = 'first'), categorical)
    ]
)
X_train = ct.fit_transform(X_train)
X_test = ct.transform(X_test)

X_train.shape, X_test.shape

In [ ]:
import sklearn.preprocessing

ss = sklearn.preprocessing.StandardScaler()
X_train_scaled = ss.fit_transform(X_train)
X_test_scaled = ss.transform(X_test)